In [ ]:
# -*- coding: utf-8 -*-
"""
Auteur : Kalonji
Date : 2026
"""

# Import des librairies PyTorch et Torchvision
import torch                      # pour le calcul tensoriel et GPU
import torch.nn as nn             # pour les réseaux de neurones
import torch.optim as optim       # pour les optimizers
from torch.utils.data import DataLoader   # pour gérer les batches
import torchvision.datasets as dset        # pour télécharger CIFAR-10
import torchvision.transforms as transforms # pour transformer les images
import torchvision.utils as vutils          # pour sauvegarder les images générées
import os                        # pour gérer les dossiers

# ==============================
# Paramètres principaux
# ==============================
dataroot = "./data"        # dossier pour stocker le dataset CIFAR-10
batch_size = 128           # nombre d'images par batch
image_size = 64            # taille des images de sortie du DCGAN
nc = 3                     # nombre de canaux (3 pour RGB)
nz = 100                   # dimension du vecteur latent d'entrée du générateur
ngf = 64                   # taille de base pour le générateur
ndf = 64                   # taille de base pour le discriminateur
num_epochs = 5             # nombre d'epochs pour l'entraînement
lr = 0.0002                # learning rate pour Adam optimizer
beta1 = 0.5                # paramètre beta1 pour Adam
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # utilisation GPU si disponible

# création du dossier pour stocker les images générées
os.makedirs("generated_images", exist_ok=True)

# ==============================
# Préparation du dataset CIFAR-10
# ==============================
dataset = dset.CIFAR10(
    root=dataroot,             # dossier pour télécharger les données
    download=True,             # télécharger si non présent
    transform=transforms.Compose([  # transformation des images
        transforms.Resize(image_size),  # redimensionner les images en 64x64
        transforms.CenterCrop(image_size),  # recadrer le centre
        transforms.ToTensor(),             # convertir en tensor
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # normaliser entre -1 et 1
    ])
)

# Création du DataLoader pour parcourir les images par batch
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# ==============================
# Définition du Générateur
# ==============================
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        # Sequence de couches ConvTranspose2d pour générer des images
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf*8, 4, 1, 0, bias=False),  # input Z->4x4 feature map
            nn.BatchNorm2d(ngf*8),    # normalisation pour stabiliser l'entraînement
            nn.ReLU(True),            # activation ReLU pour non-linéarité

            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False), # upscale 4x4 -> 8x8
            nn.BatchNorm2d(ngf*4),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False), # 8x8 -> 16x16
            nn.BatchNorm2d(ngf*2),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),   # 16x16 -> 32x32
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),      # 32x32 -> 64x64
            nn.Tanh()   # activation Tanh pour sortir des valeurs entre -1 et 1
        )

    def forward(self, input):
        return self.main(input)  # passage des données à travers le réseau

# ==============================
# Définition du Discriminateur
# ==============================
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        # Sequence de couches Conv2d pour classifier les images
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),  # input 64x64 -> 32x32
            nn.LeakyReLU(0.2, inplace=True),           # LeakyReLU pour ne pas tuer les gradients

            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False), # 32x32 -> 16x16
            nn.BatchNorm2d(ndf*2),                     # normalisation
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False), # 16x16 -> 8x8
            nn.BatchNorm2d(ndf*4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False), # 8x8 -> 4x4
            nn.BatchNorm2d(ndf*8),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False),  # 4x4 -> 1x1
            nn.Sigmoid()  # sortie entre 0 et 1 (probabilité réel/faux)
        )

    def forward(self, input):
        return self.main(input)  # passage des données à travers le réseau

# ==============================
# Initialisation des réseaux et optimizers
# ==============================
netG = Generator().to(device)  # générateur sur GPU/CPU
netD = Discriminator().to(device)  # discriminateur sur GPU/CPU

criterion = nn.BCELoss()  # Binary Cross Entropy pour la classification réel/faux
real_label = 1.            # label pour les vraies images
fake_label = 0.            # label pour les fausses images

# Optimizers Adam
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))  # pour le Discriminateur
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))  # pour le Générateur

# ==============================
# Boucle d'entraînement
# ==============================
print("Début de l'entraînement...")
for epoch in range(num_epochs):  # boucle sur les epochs
    for i, data in enumerate(dataloader, 0):  # boucle sur les batches
        ############################
        # Update du Discriminateur
        ############################
        netD.zero_grad()  # reset des gradients
        real_images = data[0].to(device)  # récupération des images réelles
        b_size = real_images.size(0)      # taille du batch
        label = torch.full((b_size,), real_label, device=device)  # label réel

        output = netD(real_images).view(-1)  # prédiction du discriminateur
        errD_real = criterion(output, label)  # calcul de la loss pour les vraies images
        errD_real.backward()  # backpropagation pour le réel

        # Génération des images fausses avec le Générateur
        noise = torch.randn(b_size, nz, 1, 1, device=device)  # vecteur latent aléatoire
        fake_images = netG(noise)  # images générées
        label.fill_(fake_label)     # label faux pour le fake
        output = netD(fake_images.detach()).view(-1)  # prédiction du discriminateur
        errD_fake = criterion(output, label)  # calcul de la loss pour les fausses images
        errD_fake.backward()  # backpropagation pour le fake
        optimizerD.step()     # mise à jour des poids du Discriminateur

        ############################
        # Update du Générateur
        ############################
        netG.zero_grad()       # reset des gradients
        label.fill_(real_label)  # on veut tromper le discriminateur
        output = netD(fake_images).view(-1)  # prédiction du discriminateur
        errG = criterion(output, label)      # loss pour le générateur
        errG.backward()       # backpropagation
        optimizerG.step()     # mise à jour des poids du Générateur

        # Affichage des losses tous les 100 batches
        if i % 100 == 0:
            print(f"[{epoch}/{num_epochs}][{i}/{len(dataloader)}] "
                  f"Loss_D: {errD_real.item()+errD_fake.item():.4f} "
                  f"Loss_G: {errG.item():.4f}")

    # Sauvegarde des images générées à chaque epoch
    vutils.save_image(fake_images.data[:64],
                      f"generated_images/epoch_{epoch}.png",
                      normalize=True)

print("Entraînement terminé !")

# ==============================
# Génération finale après entraînement
# ==============================
with torch.no_grad():  # pas de calcul des gradients
    fixed_noise = torch.randn(64, nz, 1, 1, device=device)  # vecteur latent fixe
    fake_images = netG(fixed_noise)  # images générées finales
    vutils.save_image(fake_images, "generated_images/final_generated.png", normalize=True)

print("Images générées sauvegardées dans ./generated_images/")